# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MirAziz0/flyrank1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Type: scoring / ranking, built on top of a binary classifier.**

The final deliverable (from ML-02) is a ranked queue: "which pages should a reviewer look at first." That is fundamentally a **ranking** problem — the output is an ordering, not a single yes/no answer. But the way the starter pipeline (and my project) gets there is through **binary classification**: train a model to output `P(page is declining)` for each content item, then combine that probability with a normalized rule-based score into one number (`final_refresh_score`) and sort by it. So the task type is best described in two layers:

- **Underlying model:** binary classification (`is_declining_label` ∈ {0, 1}).
- **What the output is used for:** scoring and ranking — sorting all eligible pages by predicted risk/opportunity so reviewer time goes to the top of the list first.

It is not clustering (I am not looking for unlabeled groups of similar pages) and it is not pure classification in the sense of "flag/don't flag" — the probability itself, and its rank position, matter more than a hard 0/1 cutoff.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

TASK_TYPE = "binary classification (P(declining)) -> combined into a scoring/ranking output"
print(TASK_TYPE)


binary classification (P(declining)) -> combined into a scoring/ranking output


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Starter proxy label:** `is_declining_label = (trend_direction == "down")`, exactly as computed in `scripts/01_prepare_features.py`. This is a **defined-rule proxy**, not a future observed outcome: `trend_direction` is a bucket calculated from the *current* 90-day window (comparing `impressions_last_30d`/`clicks_last_30d` against `impressions_prev_30d`/`clicks_prev_30d`), so the label describes "already moved down recently," not "will decline next month."

I am using this starter proxy for this week's framing because it is the label the committed pipeline and its verified results (`outputs/model_report.md`) are built on, and it lets me show a real dataframe and real numbers now. But I'm flagging its weakness honestly, per the lane guide: a **stronger capstone target** would be a genuine future-observed outcome, built with a strict feature/target window split, e.g.:

```text
features from prior 90 days -> decline (or recovery) over the next 30 days
```

That version needs `report_date`-indexed daily facts (from the warehouse release) rather than the single-snapshot starter CSV, and a leakage audit to make sure no next-30-day metric leaks into the features. I'll build that once I move to warehouse data in Week 3+; for now the proxy is clearly labeled as a proxy, not treated as ground truth.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates(subset=["content_id"]).reset_index(drop=True)

df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("Target definition: is_declining_label = (trend_direction == 'down')")
print(df["is_declining_label"].value_counts())
print("Positive rate:", round(df["is_declining_label"].mean(), 3))


Target definition: is_declining_label = (trend_direction == 'down')
is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Positive rate: 0.542


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric: Precision@50** (with average precision as a secondary check).

Why this one and not plain accuracy or ROC AUC alone: the output is consumed as a **ranked list that a reviewer works through from the top**, and the reviewer has a fixed weekly capacity — the lane guide's own example uses 50 pages/week. Precision@50 answers exactly the question that matters operationally: *"of the top 50 pages the system tells someone to review first, how many are actually right?"* That ties the metric directly to the real decision (Section 2 of ML-02), instead of measuring performance on cases nobody will ever act on.

- Accuracy is a poor fit here: the label is fairly balanced (~54% positive) so accuracy would look "good" even from a naive rule, and it says nothing about the *top* of the ranking, which is the only part anyone reads.
- ROC AUC is useful as a general discrimination check (and I'll still report it), but it treats every rank position equally, when in reality position 1 vs. position 50 matters far more than position 4,000 vs 4,050.
- Average precision is a good secondary metric — it summarizes the *whole* ranking's quality, useful if the reviewer capacity changes.

Good already means something concrete here: the committed starter run shows baseline rules at **Precision@50 = 0.240** and random forest at **Precision@50 = 0.740** (`outputs/model_report.md`) — so "good" for my own model means clearing the baseline by a real margin, not just beating a coin flip.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

def precision_at_k(scores_desc_labels, k):
    top_k = scores_desc_labels[:k]
    return sum(top_k) / k

# Committed, verified pipeline results (outputs/model_report.md) — not re-trained in this notebook.
precision_at_50 = {"baseline_rules": 0.240, "random_forest": 0.740}
print("Chosen success metric: Precision@50")
print("Verified starter results:", precision_at_50)
print(f"Baseline gets ~{precision_at_50['baseline_rules']*50:.0f}/50 right; "
      f"random forest gets ~{precision_at_50['random_forest']*50:.0f}/50 right.")


Chosen success metric: Precision@50
Verified starter results: {'baseline_rules': 0.24, 'random_forest': 0.74}
Baseline gets ~12/50 right; random forest gets ~37/50 right.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content item (`content_id`) belonging to one client (`client_id`), summarized over its trailing 90-day window** — after filtering to pages with `impressions_90d > 0` and `content_age_days >= 90` (the same eligibility filter the starter pipeline uses) and de-duplicating by `content_id`. It is not a query, not a day, not a client: each row is a single page a reviewer could open and act on.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Shape (rows, cols):", df.shape)
print("One row =", df["content_id"].is_unique, "-> content_id is unique per row" if df["content_id"].is_unique else "duplicates present")

unit_of_analysis_cols = [
    "content_id", "client_id", "content_type", "main_intent",
    "impressions_90d", "clicks_90d", "sessions_90d", "avg_position",
    "content_age_days", "days_since_last_update", "trend_direction", "is_declining_label",
]
df[unit_of_analysis_cols].head(8)


Shape (rows, cols): (30000, 45)
One row = True -> content_id is unique per row


,content_id,client_id,content_type,main_intent,impressions_90d,clicks_90d,sessions_90d,avg_position,content_age_days,days_since_last_update,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3803,29,17,10.6,187,20,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,15320,7,9,20.3,445,25,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,12581,11,11,36.5,141,20,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,11751,58,78,6.2,463,22,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,19140,24,145,44.0,263,14,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,transactional,3970,1,5,8.5,147,20,down,1
6,content_9a34b442b552,client_8722616204,keyword article,informational,20,0,1,7.0,90,20,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,commercial,1724,1,28,21.2,445,22,stable,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

No single threshold rule captures "declining with demand" well on its own — each individual signal (impressions, position, freshness, engagement) only weakly correlates with the label by itself (see the code cell below), because a page can decline for different, overlapping reasons: it can be visible but stale, well-positioned but low-engagement, high-demand but thin content, and so on. A hand-written rule can only combine a few conditions with AND/OR before it becomes an unreadable, brittle nest of thresholds tuned by trial and error — and the starter baseline (`baseline_refresh_score`, a fixed weighted combination of 4 sub-scores) already shows the ceiling of that approach: **Precision@50 = 0.240**.

A model (random forest here) can learn nonlinear interactions between many weak signals at once — e.g. "low CTR only matters when position is good AND impressions are high" — without anyone hand-coding every combination. That is exactly why the committed run more than triples precision at the top of the queue (**0.240 → 0.740**): the *signal* was distributed across many correlated-but-individually-weak features, which is the textbook case where ML earns its complexity over a fixed rule.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

candidate_signals = ["impressions_90d", "avg_position", "days_since_last_update", "ctr", "engagement_rate", "scroll_rate", "word_count"]
correlations = df[candidate_signals + ["is_declining_label"]].corr(numeric_only=True)["is_declining_label"].drop("is_declining_label")
print("Correlation of each individual signal with is_declining_label:")
print(correlations.sort_values(key=abs, ascending=False).round(3))
print()
print("Every single signal is weak on its own (|r| well under 0.3) -> no one threshold separates the classes cleanly.")


Correlation of each individual signal with is_declining_label:
word_count                0.090
days_since_last_update    0.081
ctr                      -0.062
avg_position             -0.029
impressions_90d          -0.018
engagement_rate          -0.013
scroll_rate              -0.003
Name: is_declining_label, dtype: float64

Every single signal is weak on its own (|r| well under 0.3) -> no one threshold separates the classes cleanly.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.